# D68 LLMOps Airflow Code

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.11

# 문서 갱신 파이프라인의 Airflow 배포 실습

## 상황과 오늘 할 일

사내 규정 검색 서비스는 문서를 수집하고 정제한 뒤 인덱스를 만들고 평가합니다. 평가에 실패한 후보가 운영에 반영되지 않도록, 이 순서를 Airflow의 작업으로 연결합니다. 실습 자료는 사내 규정을 가정한 합성 문서이며 실제 고객정보를 사용하지 않습니다.

문서를 처리하는 함수는 제공됩니다. 여러분은 `@dag`와 `@task`로 실행 일정과 작업 간 의존 관계를 구성하고, Airflow UI에서 직접 실행한 태스크의 상태와 전달값, 로그를 확인합니다. 이후 정제 기준 때문에 평가가 실패하는 상황을 실행해 승격이 차단되는지 확인하고 기준을 수정합니다.

## 먼저 알아둘 개념

**Airflow**는 여러 작업을 일정과 의존 관계에 따라 실행하고 작업별 상태를 기록하는 도구입니다. `@task`는 파이썬 함수를 태스크로 만들고 `@dag`는 태스크들을 하나의 DAG로 묶습니다. DAG는 순환 없이 연결한 작업 그래프이며, 앞 태스크의 반환값을 뒤 태스크에 전달하면 두 태스크 사이의 의존 관계도 생깁니다.

**LLMOps**는 LLM 서비스의 구성과 변경, 평가와 배포를 반복 가능하게 관리하는 방식입니다. 오늘은 모델을 학습하거나 호출하지 않고 문서 갱신 작업을 연결합니다. 함께 검증한 인덱스·프롬프트 등의 버전을 출시 번들로 묶고, `current`가 운영에서 사용할 번들을 가리키도록 합니다.

## 학습목표

- @dag·@task로 실행 일정·재시도·태스크 연결을 작성할 수 있다.
- Airflow UI의 Graph·Grid·Logs·Storage(XCom)에서 실제 실행과 실패 근거를 확인할 수 있다.
- 실패 태스크의 Clear와 설정을 수정한 새 실행을 비교하고 운영 참조의 유지 여부를 검증할 수 있다.

## 진행 순서와 시간

| 시간 | 활동 | 분 |
|---|---|---:|
| 10:00~10:45 | LLMOps·Airflow 개념과 실행 흐름 | 45 |
| 10:45~11:05 | 휴식 | 20 |
| 11:05~12:00 | 환경 준비·DAG 작성·의존 관계 확인 | 55 |
| 12:00~13:00 | 정상 번들 사전 점검·UI 접속·Graph와 XCom 확인 | 60 |
| 13:00~14:00 | 점심 | 60 |
| 14:00~15:20 | UI Trigger·Logs·Clear·설정 수정 후 새 실행 | 80 |
| 15:20~15:40 | 휴식 | 20 |
| 15:40~16:40 | UI 기록·복구 범위 작성·동료 대조 | 60 |
| 16:40~17:00 | 결과 저장·Airflow 종료·질문 정리 | 20 |

## 0. 실습 준비

이번 실습은 로컬 PC의 Docker Linux 컨테이너에서 진행합니다. JupyterLab과 Airflow를 같은 컨테이너에서 실행하고, 브라우저의 서로 다른 탭에서 사용합니다. GPU와 모델 다운로드는 필요하지 않습니다.

노트북에 포함된 준비 코드가 Docker 실행 파일을 생성하므로 별도 코드 저장소를 내려받을 필요가 없습니다. 아래 안내에 따라 로컬 터미널에서 준비한 뒤, 컨테이너의 JupyterLab에 접속해 「0.2」부터 실행합니다. 문서 처리 함수는 준비 셀에서 생성하며, DAG의 실행 일정과 태스크 연결은 「1」에서 직접 작성합니다.

### 0.1 로컬 실행 파일 생성

JupyterLab과 Airflow를 같은 Linux 컨테이너에서 실행합니다. 코드 작성은 JupyterLab에서, 실행 상태·로그·XCom 확인은 별도 Airflow 탭에서 진행합니다. CPU만 사용합니다.

#### 처음 실행할 때

1. 컨테이너 심화와 쿠버네티스 노트(D67)에서 사용한 Docker를 실행합니다. Windows에서는 Docker Desktop이 Linux 컨테이너 모드인지 확인합니다. Docker에 메모리를 최소 4GB 할당합니다.
2. 로컬 PC에 `D68_실습` 폴더를 만들고 이 노트북을 넣습니다. PowerShell 또는 터미널에서 해당 폴더를 엽니다. 개인 파일이 함께 있는 다운로드 폴더 전체를 실습 폴더로 사용하지 않습니다.
3. `docker version`과 `docker compose version`으로 실행 환경을 확인합니다. 이어서 아래 명령을 실행하면 노트북의 준비 코드가 `d68_local`에 Docker 실행 파일을 생성합니다. Linux·macOS에서 `python` 명령이 없다면 `python3`으로 실행합니다.

```shell
python -c "import json; from pathlib import Path; n=json.loads(Path('D68_LLMOps_Airflow_Code.ipynb').read_text(encoding='utf-8')); exec(''.join(next(c for c in n['cells'] if c.get('metadata',{}).get('d68_role')=='docker_bootstrap')['source']))"
```

4. 같은 터미널에서 다음 명령을 실행합니다. 첫 실행에는 이미지 다운로드와 패키지 설치가 포함됩니다.

```shell
cd d68_local
docker compose up --build -d
docker compose ps
```

5. `notebook`의 상태가 `healthy`가 되면 준비 코드가 출력한 `http://localhost:8888/…?token=…` 주소를 브라우저에서 엽니다. JupyterLab의 토큰을 묻는 경우 `d68_local/.env`의 `D68_TOKEN` 값을 입력합니다. 토큰은 각자의 로컬 환경에서 생성됩니다.
6. 컨테이너의 JupyterLab에서 `D68_LLMOps_Airflow_Code.ipynb`를 열고 「0.2 컨테이너 환경 확인」부터 실행합니다. 「0.1 로컬 실행 파일 생성」은 다시 실행해도 준비 완료 안내만 출력합니다. DAG 작성 셀에는 직접 채울 다섯 칸이 있으므로, 「1.2」에서 여섯 태스크가 출력되는지 확인한 뒤 진행합니다.
7. 「3.1」에서 Airflow를 기동한 뒤 브라우저의 새 탭에서 `http://localhost:8680/dags`를 엽니다. Docker만 켠 상태에서는 Airflow가 아직 시작되지 않습니다.

#### 실행 위치와 저장 위치

| 작업 | 실행할 곳 | 주소 또는 경로 |
|---|---|---|
| Docker 기동·상태 확인·종료 | 로컬 PC의 터미널, `d68_local` 폴더 | `docker compose …` |
| 파이썬 코드와 DAG 작성 | 컨테이너의 JupyterLab | `http://localhost:8888` |
| Trigger·Graph·Logs·Storage(XCom) | 로컬 PC의 브라우저, Airflow 탭 | `http://localhost:8680/dags` |
| 노트북과 결과 파일 보관 | 로컬 PC의 `D68_실습` 폴더 | 컨테이너에서는 `/workspace`로 보임 |
| DAG 파일 | 노트북의 작성 셀에서 생성 | `work_d68/dags/d68_weekly_refresh.py` |

같은 폴더를 연결하는 **바인드 마운트**로 노트북과 결과 파일을 공유합니다. 컨테이너의 `/workspace/work_d68`는 로컬 실습 폴더의 `work_d68`와 같은 파일을 가리킵니다. Windows 기본 Python 커널이나 Colab에서 본 실습을 이어 실행하지 않습니다.

#### 접속이 안 될 때

- JupyterLab이 열리지 않으면 `docker compose ps`와 `docker compose logs --tail 40 notebook`을 확인합니다. 토큰 주소를 잊었다면 `docker compose exec notebook python -m jupyter server list`로 다시 확인합니다.
- `8888` 또는 `8680` 포트가 이미 사용 중이면 기존에 실행한 같은 실습을 종료한 뒤 다시 기동합니다. 이 교안은 두 포트를 고정해서 사용합니다.
- Airflow가 열리지 않으면 「3.1」의 준비 완료 메시지와 `work_d68/standalone.log`를 확인합니다. Airflow 목록에 DAG가 없으면 먼저 「1.2」의 오류를 해결하고 파일 경로와 두 DAG ID를 확인합니다.
- 커널만 재시작한 뒤 Airflow 포트가 이미 사용 중이라고 나오면, 노트북을 저장하고 호스트 터미널에서 `docker compose restart notebook`을 실행합니다. JupyterLab에 다시 접속해 「0.2」부터 진행합니다.

#### 마칠 때와 다시 시작할 때

노트북의 제출 셀과 `stop_airflow()`를 실행하고 노트북을 저장합니다. 로컬 실습 폴더에서 `work_d68/d68_submission.json`과 로그를 확인한 뒤, 호스트 터미널의 `d68_local` 폴더에서 다음을 실행합니다.

```shell
docker compose down
```

컨테이너를 종료해도 바인드 마운트로 연결한 로컬 파일은 남습니다. 다음에 같은 실습을 이어 할 때는 `docker compose up -d`를 실행하고 JupyterLab에 접속합니다.

In [17]:
import os
import secrets
from pathlib import Path

NOTEBOOK_NAME = "D68_LLMOps_Airflow_Code.ipynb"

if os.environ.get("D68_DOCKER") == "1":
    print("Docker의 JupyterLab에 접속했습니다. 다음 환경 확인 셀부터 실행하세요.")
else:
    lesson_dir = Path.cwd().resolve()
    if not (lesson_dir / NOTEBOOK_NAME).is_file():
        raise FileNotFoundError(
            f"{NOTEBOOK_NAME}이 있는 실습 폴더에서 준비 코드를 실행하세요. 현재 폴더: {lesson_dir}"
        )
    setup_dir = lesson_dir / "d68_local"
    setup_dir.mkdir(exist_ok=True)

    dockerfile = '''FROM apache/airflow:3.3.2-python3.12
USER airflow
RUN pip install --no-cache-dir "apache-airflow==3.3.2" "jupyterlab>=4,<5" \\
    --constraint https://raw.githubusercontent.com/apache/airflow/constraints-3.3.2/constraints-no-providers-3.12.txt
WORKDIR /workspace
'''
    compose = '''name: d68-local
services:
  notebook:
    build:
      context: .
    user: "${D68_UID:-50000}:0"
    init: true
    working_dir: /workspace
    environment:
      D68_DOCKER: "1"
      JUPYTER_TOKEN: "${D68_TOKEN:?준비 코드를 먼저 실행하세요}"
      JUPYTER_RUNTIME_DIR: /tmp/d68-jupyter-runtime
      JUPYTER_DATA_DIR: /tmp/d68-jupyter-data
      JUPYTER_CONFIG_DIR: /tmp/d68-jupyter-config
      IPYTHONDIR: /tmp/d68-ipython
      CHECK_MAX_COUNT: "0"
      PYTHONUNBUFFERED: "1"
    ports:
      - "127.0.0.1:8888:8888"
      - "127.0.0.1:8680:8680"
    volumes:
      - type: bind
        source: ..
        target: /workspace
    command:
      - python
      - -m
      - jupyterlab
      - --ip=0.0.0.0
      - --port=8888
      - --no-browser
      - --ServerApp.port_retries=0
      - --ServerApp.root_dir=/workspace
      - --ServerApp.default_url=/lab/tree/D68_LLMOps_Airflow_Code.ipynb
      - --ServerApp.use_redirect_file=False
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8888/login', timeout=5)"]
      interval: 10s
      timeout: 6s
      retries: 12
      start_period: 30s
'''
    (setup_dir / "Dockerfile").write_text(dockerfile, encoding="utf-8")
    (setup_dir / "compose.yaml").write_text(compose, encoding="utf-8")
    (setup_dir / ".dockerignore").write_text("*\n!Dockerfile\n", encoding="utf-8")

    # 재실행 시 접속 토큰과 사용자 설정을 유지합니다.
    env_file = setup_dir / ".env"
    if not env_file.exists():
        uid = os.getuid() if hasattr(os, "getuid") else 50000
        env_file.write_text(
            f"D68_UID={uid}\nD68_TOKEN={secrets.token_urlsafe(24)}\n", encoding="utf-8"
        )
    values = dict(
        line.split("=", 1) for line in env_file.read_text(encoding="utf-8").splitlines()
        if "=" in line and not line.startswith("#")
    )
    token = values["D68_TOKEN"]
    print("준비 파일 위치:", setup_dir)
    print("같은 터미널에서 차례대로 실행하세요:\ncd d68_local\ndocker compose up --build -d\ndocker compose ps")
    print("JupyterLab 접속 주소:")
    print(f"http://localhost:8888/lab/tree/{NOTEBOOK_NAME}?token={token}")
    print("컨테이너 상태가 healthy가 되면 위 주소를 브라우저에서 여세요.")
    print("Airflow는 노트북 3.1절에서 기동한 뒤 http://localhost:8680/dags 로 접속합니다.")

Docker의 JupyterLab에 접속했습니다. 다음 환경 확인 셀부터 실행하세요.


### 0.2 컨테이너 환경 확인

이 셀부터는 로컬 브라우저의 `http://localhost:8888`에 연결한 JupyterLab에서 실행합니다. 준비된 이미지의 Python과 Airflow 버전을 확인합니다. 설치 셀을 반복 실행하거나 Windows 기본 Python 환경에 Airflow를 설치하지 않습니다.

In [18]:
import os
import sys
from importlib.metadata import version

if os.environ.get("D68_DOCKER") != "1" or sys.platform != "linux":
    raise RuntimeError(
        "본 실습은 Docker의 JupyterLab에서 실행합니다. 0.1절의 기동 절차를 마친 뒤 "
        "http://localhost:8888 에 접속해 이 노트북을 여세요."
    )
AIRFLOW_VERSION = version("apache-airflow")
if AIRFLOW_VERSION != "3.3.2":
    raise RuntimeError(f"Airflow 3.3.2 이미지가 필요합니다. 현재 버전: {AIRFLOW_VERSION}")
print("Python:", sys.version.split()[0], "/ Airflow:", AIRFLOW_VERSION)
print("실행 위치: Docker Linux 컨테이너 / 실습 파일: /workspace")

Python: 3.12.14 / Airflow: 3.3.2
실행 위치: Docker Linux 컨테이너 / 실습 파일: /workspace


### 0.3 문서 처리 함수와 실습 자료 생성

바인드 마운트로 연결한 실습 폴더를 `/workspace`에서 사용하고 그 아래에 `work_d68`를 만듭니다. `d68_pipeline.py`는 수집·정제·청크 생성을, `d68_release.py`는 인덱스 생성·평가·운영 참조 변경을 담당합니다. 두 파일은 제공되는 일반 파이썬 함수이며, 이후 작성할 DAG의 태스크가 호출합니다.

In [19]:
import os
import sys
from pathlib import Path

# 다시 실행해도 같은 작업 폴더를 사용합니다.
COURSE_DIR = Path("/workspace")
WORK = COURSE_DIR / "work_d68"
DAGS = WORK / "dags"
DAGS.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)                          # %%writefile 로 만드는 파일이 이 폴더 아래에 생깁니다
if str(DAGS) not in sys.path:           # DAG 파일과 함수 파일을 import 할 수 있게 합니다
    sys.path.insert(0, str(DAGS))

os.environ.update({
    "D68_WORK": str(WORK),
    "AIRFLOW_HOME": str(WORK / "airflow"),
    "AIRFLOW__CORE__DAGS_FOLDER": str(DAGS),
    "AIRFLOW__CORE__LOAD_EXAMPLES": "False",          # 예제 DAG 수십 개를 로드하지 않습니다
    "AIRFLOW__LOGGING__LOGGING_LEVEL": "WARNING",     # dag.test() 출력이 화면을 덮지 않게 합니다
    "AIRFLOW__CORE__PARALLELISM": "4",                # standalone 워커 수 (기본 32)
})
print("작업 폴더", WORK, "\nAIRFLOW_HOME", os.environ["AIRFLOW_HOME"])
os.environ["AIRFLOW__CORE__DAG_RUN_CONF_OVERRIDES_PARAMS"] = "True"

작업 폴더 /workspace/work_d68 
AIRFLOW_HOME /workspace/work_d68/airflow


In [20]:
%%writefile dags/d68_pipeline.py
"""문서 수집·정제·청크 생성 함수입니다. Airflow를 import하지 않습니다"""
import hashlib
import json
import os
from pathlib import Path
from zoneinfo import ZoneInfo

WORK = Path(os.environ.get("D68_WORK", "/content/d68"))
DRIVE = WORK / "corp_docs"      # 공유 드라이브 역할
BLOBS = WORK / "blobs"          # 문서 내용을 해시 이름으로 한 벌만 보관
SNAPS = WORK / "snapshots"      # 그날의 문서 목록
PARTS = WORK / "parts"          # 날짜 파티션 : 청크와 평가 보고서
TZ = ZoneInfo(os.environ.get("D68_TZ", "Asia/Seoul"))


class DriveUnavailable(ConnectionError):
    """공유 드라이브가 잠깐 끊김. 다시 하면 될 수 있습니다"""


class BadConfig(ValueError):
    """설정 값이 잘못됨. 다시 해도 같습니다"""


class MissingSnapshot(FileNotFoundError):
    """백필하는 날의 목록이 없음. 다시 해도 같습니다"""


def fp(data, n=12):                     # 내용에서 계산한 해시 (파이프라인 노트 2.3절)
    return hashlib.sha256(data.encode("utf-8") if isinstance(data, str) else data).hexdigest()[:n]


def write_atomic(path, text):           # 다 쓴 뒤 이름을 바꿔 반쯤 쓴 파일이 남지 않게 합니다
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)
    return str(path)


def day_of(logical_date):               # 논리적 날짜(UTC) -> 한국 날짜 = 파티션 이름
    return logical_date.astimezone(TZ).date().isoformat()


def _maybe_flaky():                     # 실습용 : D68_FLAKY=N 이면 처음 N번은 드라이브가 끊긴 것처럼
    counter = WORK / ".flaky_count"
    n = int(counter.read_text()) if counter.exists() else 0
    if n < int(os.environ.get("D68_FLAKY", "0")):
        counter.write_text(str(n + 1))
        raise DriveUnavailable(f"공유 드라이브 응답 없음 ({n + 1}번째 시도)")


def stage_catalog(day, allow_scan=True):
    """1단계 목록화. 그날의 문서 목록과 해시를 남기고 그 경로를 반환합니다"""
    snap = SNAPS / f"{day}.json"
    if snap.exists():                    # 같은 날을 다시 실행하면 같은 입력을 씁니다
        return str(snap)
    if not allow_scan:                   # 백필하는 날인데 그날 목록이 없습니다
        raise MissingSnapshot(f"{day} 목록이 없습니다. 지금 드라이브로 대신하지 않습니다")
    _maybe_flaky()
    files = sorted(p for p in DRIVE.rglob("*") if p.is_file()) if DRIVE.exists() else []
    if not files:                        # 연결이 빠진 드라이브는 빈 폴더로 보입니다
        raise DriveUnavailable("공유 드라이브가 비어 있습니다. 연결을 확인하세요")
    BLOBS.mkdir(parents=True, exist_ok=True)
    rows = []
    for p in files:
        data = p.read_bytes()
        sha = fp(data, 64)
        if not (BLOBS / sha).exists():
            (BLOBS / sha).write_bytes(data)
        rows.append({"path": p.relative_to(DRIVE).as_posix(), "sha": sha, "size": len(data)})
    return write_atomic(snap, json.dumps({"day": day, "files": rows}, ensure_ascii=False, indent=1))


def stage_filter(doc, min_chars):       # 3단계 필터링. 빼는 이유를 반환합니다 (남기면 "")
    body = doc["text"].strip()
    if doc["path"].startswith("보관/"):
        return "보관: 옛 버전"
    if len(body) < min_chars:
        return f"짧음: {len(body)}자"
    return "서식만 있음" if "내용을 입력하세요" in body else ""


def stage_chunk(doc, chunk_size):       # 8단계 청킹. 조항(##) 단위로 자르고 chunk_size 자를 넘으면 다시 자릅니다
    title = doc["text"].strip().split("\n", 1)[0].lstrip("# ").strip()
    secs = ("\n" + doc["text"]).split("\n## ")
    secs = ["## " + s for s in secs[1:]] if len(secs) > 1 else [doc["text"]]
    out = []
    for sec in secs:
        for i in range(0, len(sec), chunk_size):
            piece = sec[i:i + chunk_size].strip()
            if piece:
                out.append({"chunk_id": fp(doc["path"] + "|" + piece), "source_path": doc["path"],
                            "text": f"[{title}] {piece}"})
    return out


def soft_chunk_drift(day, n_chunks, tolerance=0.15):
    """소프트 게이트 : 바로 앞 파티션보다 청크 수가 크게 변하면 경고만 남깁니다"""
    earlier = sorted(p.parent.name for p in PARTS.glob("*/chunks.json") if p.parent.name < day)
    if not earlier:
        return []
    before = len(json.loads((PARTS / earlier[-1] / "chunks.json").read_text(encoding="utf-8"))["chunks"])
    if before and abs(n_chunks - before) / before > tolerance:
        return [f"청크 {before} -> {n_chunks}개 ({earlier[-1]} 대비). 제외 사유별 건수를 먼저 확인하세요"]
    return []


def prepare(snapshot_path, params, min_rate=0.3, *, parts_dir=None):
    """2~8단계(수집, 필터링, 중복 제거, 청킹)를 게이트와 함께 실행합니다.
    게이트에서 멈추면 예외 대신 결과로 알립니다"""
    chunk_size, min_chars = int(params.get("chunk_size", 400)), int(params.get("min_chars", 10))
    if chunk_size < 50 or min_chars < 0:
        raise BadConfig(f"chunk_size={chunk_size}, min_chars={min_chars} 는 쓸 수 없는 값입니다")
    snap = json.loads(Path(snapshot_path).read_text(encoding="utf-8"))
    docs = [dict(row, text=(BLOBS / row["sha"]).read_text(encoding="utf-8"), why="") for row in snap["files"]]
    for d in docs:
        d["why"] = stage_filter(d, min_chars)
    counts = {"ingest": len(docs), "filter": sum(not d["why"] for d in docs)}
    if counts["filter"] / max(len(docs), 1) < min_rate:          # 하드 게이트 : 통과율
        msg = f"통과율 {counts['filter'] / max(len(docs), 1):.0%}가 기준 {min_rate:.0%} 미만입니다"
        return {"ok": False, "stopped_at": "filter", "message": msg, "counts": counts}
    first = {}                                                   # 4단계 중복 제거 : 경로가 앞선 것만 남김
    for d in sorted(docs, key=lambda d: d["path"]):
        if not d["why"]:
            d["why"] = f"중복: {first[d['sha']]}" if d["sha"] in first else ""
            first.setdefault(d["sha"], d["path"])
    counts["dedup"] = sum(not d["why"] for d in docs)
    chunks = [c for d in docs if not d["why"] for c in stage_chunk(d, chunk_size)]
    out = {
        "day": snap["day"],
        "snapshot": fp(json.dumps(snap["files"], sort_keys=True)),   # 날짜가 아니라 파일 목록의 해시
        "config": {"chunk_size": chunk_size, "min_chars": min_chars},
        "counts": counts,
        "warnings": soft_chunk_drift(snap["day"], len(chunks)),
        "excluded": [{"path": d["path"], "reason": d["why"]} for d in docs if d["why"]],
        "chunks": chunks,
    }
    path = write_atomic((parts_dir or PARTS) / snap["day"] / "chunks.json", json.dumps(out, ensure_ascii=False, indent=1))
    return {"ok": True, "chunks_path": path, "counts": counts, "warnings": out["warnings"]}

Overwriting dags/d68_pipeline.py


In [21]:
%%writefile dags/d68_release.py
"""검색 인덱스·평가·운영 참조 관리 함수입니다. Airflow를 import하지 않습니다"""
import json
import math
import os
import time
from pathlib import Path

from d68_pipeline import PARTS, WORK, fp, write_atomic

INDEXES = WORK / "indexes"           # 인덱스 버전마다 폴더 하나
RELEASES = WORK / "releases"         # 출시 번들과 current 별칭
CURRENT = RELEASES / "current.json"
EVAL_SET = Path(os.environ.get("D68_EVAL_SET", str(WORK / "eval" / "규정-질문-v2.json")))
PROMPT = Path(os.environ.get("D68_PROMPT", str(WORK / "prompts" / "규정-답변-v4.txt")))
MODEL = os.environ.get("D68_MODEL", "Qwen/Qwen2.5-7B-Instruct")   # 오늘은 로드하지 않고 버전만 적습니다
ADAPTER = os.environ.get("D68_ADAPTER", "")
RETRIEVER = "bigram-v1"              # 검색 방식의 이름. 바꾸면 인덱스 이름이 바뀝니다
MIN_HIT_FIRST = int(os.environ.get("D68_MIN_HIT_FIRST", "9"))      # 첫 번들이 넘어야 할 적중 수


def load(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def dump(path, obj, indent=1):
    return write_atomic(path, json.dumps(obj, ensure_ascii=False, indent=indent))


# 원문 목록이나 설정이 달라졌다면 지난 인덱스를 잘못 재사용하지 않도록 버전을 나눕니다.
def index_version(chunks_doc):          # 입력 목록과 설정의 해시가 인덱스 이름입니다
    key = {"snapshot": chunks_doc["snapshot"], "config": chunks_doc["config"], "retriever": RETRIEVER}
    return "idx-" + fp(json.dumps(key, sort_keys=True, ensure_ascii=False))


def build_index(chunks_path):
    """새 인덱스를 버전 이름의 폴더에 만듭니다. 서비스 중인 인덱스는 건드리지 않습니다"""
    doc = load(chunks_path)
    target = INDEXES / index_version(doc)
    if not (target / "index.json").exists():      # 같은 버전이 있으면 다시 만들지 않습니다
        entries = [{k: c[k] for k in ("chunk_id", "source_path", "text")} for c in doc["chunks"]]
        meta = {"index": target.name, "day": doc["day"], "snapshot": doc["snapshot"], "config": doc["config"],
                "retriever": RETRIEVER, "n_chunks": len(entries), "warnings": doc.get("warnings", [])}
        dump(target / "index.json", {"meta": meta, "entries": entries}, indent=None)
    return str(target)


def bigrams(text):
    s = "".join(ch for ch in text if not ch.isspace())
    return {s[i:i + 2] for i in range(len(s) - 1)}


def search(index_dir, query, k=3):      # 두 글자 겹침 검색. 동점은 chunk_id 로 가립니다
    q = bigrams(query)
    scored = sorted(((-len(q & bigrams(e["text"])) / math.sqrt(len(bigrams(e["text"]))), e["chunk_id"], e)
                     for e in load(Path(index_dir) / "index.json")["entries"] if bigrams(e["text"])),
                    key=lambda x: x[:2])
    return [e for _, _, e in scored[:k]]


def read_current():                     # current 별칭이 가리키는 출시 번들. 아직 없으면 None
    return load(RELEASES / f"{load(CURRENT)['release']}.json") if CURRENT.exists() else None


def evaluate(index_dir, eval_path=None, *, parts_dir=None):
    """평가 질문 세트로 새 인덱스와 current 번들의 인덱스를 함께 평가합니다"""
    eval_path, cur = Path(eval_path or EVAL_SET), read_current()

    def hit(idx_dir, q):
        return any(e["source_path"] == q["source"] for e in search(idx_dir, q["question"]))

    rows = [{"id": q["id"], "cand": hit(index_dir, q), "cur": hit(INDEXES / cur["index"], q) if cur else None}
            for q in load(eval_path)]
    meta = load(Path(index_dir) / "index.json")["meta"]
    return dump((parts_dir or PARTS) / meta["day"] / f"eval_{meta['index']}.json", {
        "index": meta["index"], "day": meta["day"], "config": meta["config"], "warnings": meta["warnings"],
        "eval_set": {"name": eval_path.stem, "fp": fp(eval_path.read_bytes())},
        "current_release": cur["release"] if cur else None,
        "cand_hit": sum(r["cand"] for r in rows),
        "cur_hit": sum(r["cur"] for r in rows) if cur else None,
        "total": len(rows),
        "flips": [r["id"] for r in rows if r["cur"] and not r["cand"]],    # 지금 맞히고 새 것이 틀림
        "rows": rows})


def decide_promotion(report_path):
    """승격할지 정합니다. 보고서를 못 읽으면 불합격입니다 (판정 함수는 불합격이 기본값)"""
    try:
        r = load(report_path)
    except (OSError, ValueError) as e:
        return False, f"평가 보고서를 읽지 못함 ({type(e).__name__})"
    if r["cur_hit"] is None:                      # 첫 번들은 정해 둔 적중 수를 넘어야 합니다
        ok = r["cand_hit"] >= MIN_HIT_FIRST
        return ok, f"첫 번들, 적중 {r['cand_hit']}/{r['total']}" if ok else \
            f"첫 번들 적중 {r['cand_hit']}/{r['total']}가 기준 {MIN_HIT_FIRST} 미만"
    if r["flips"]:
        return False, f"flip {len(r['flips'])}개 {r['flips']}"
    return True, f"적중 {r['cand_hit']}/{r['total']} (지금 {r['cur_hit']}), flip 0"


def move_current(release):              # 별칭을 다 쓴 파일로 한 번에 바꿉니다. 승격과 롤백이 같이 씁니다
    if not (RELEASES / f"{release}.json").exists():
        raise FileNotFoundError(f"{release} 번들이 없습니다")
    prev = read_current()
    dump(CURRENT, {"release": release, "previous": prev["release"] if prev else None,
                   "moved_at": time.time()}, indent=None)
    return release


def promote(report_path, airflow_run=""):
    """출시 번들을 쓰고 current 별칭을 옮깁니다. (번들 이름, 옮겼는지)를 반환합니다"""
    r = load(report_path)
    bundle = {"model": MODEL, "adapter": ADAPTER or None,
              "prompt": {"name": PROMPT.stem, "fp": fp(PROMPT.read_bytes())},
              "index": r["index"], "eval_set": r["eval_set"]}
    cur = read_current()
    if cur and all(cur.get(k) == v for k, v in bundle.items()):
        return cur["release"], False              # 이미 같은 번들을 가리킵니다
    release = f"r-{r['day'].replace('-', '')}-{fp(json.dumps(bundle, sort_keys=True, ensure_ascii=False), 6)}"
    dump(RELEASES / f"{release}.json", {"release": release, **bundle,
         "eval": {"hit": r["cand_hit"], "total": r["total"], "flips": len(r["flips"])},
         "warnings": r["warnings"], "airflow_run": airflow_run})
    return move_current(release), True


def hours_since_promotion():            # Airflow 외부 감시용 : 마지막 승격 뒤 지난 시간
    return (time.time() - load(CURRENT)["moved_at"]) / 3600 if CURRENT.exists() else None


def write_release_note(n_events):       # 에셋으로 트리거된 DAG가 새 번들의 요약을 한 줄 덧붙입니다
    cur = read_current()
    line = (f"{time.strftime('%Y-%m-%d %H:%M:%S')} 새 번들 {cur['release']} 인덱스 {cur['index']} "
            f"적중 {cur['eval']['hit']}/{cur['eval']['total']}, 받은 갱신 {n_events}건")
    with open(RELEASES / "notes.log", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    return line

Overwriting dags/d68_release.py


In [22]:
import json

import d68_pipeline as P

DRIVE = P.DRIVE
DOCS_W1 = {
    "공용/규정/복무규정_2024.md": "# 임직원 복무 규정 (2024년판)\n\n## 제3조 (근무 시간)\n기본 근무 시간은 주 40시간으로 하며, 시업 시각은 오전 9시로 한다.\n\n## 제5조 (재택근무)\n임직원은 주 1회 이내에서 재택근무를 신청할 수 있다.\n\n## 제7조 (연차 휴가의 이월)\n당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없다.\n",
    "공용/규정/보안지침.md": "# 정보보안 지침\n\n## 1. 사내 데이터의 외부 반출\n고객 개인정보, 미공개 재무 정보, 기술 설계 문서는 정보보호팀장의 사전 승인 없이 외부로 반출할 수 없다. 외부 AI 서비스에 사내 문서를 입력하는 행위도 반출로 간주한다.\n\n## 2. 위반 시 조치\n위반한 임직원은 상벌 규정에 따라 징계할 수 있다.\n",
    "공용/매뉴얼/시스템매뉴얼.md": "# 사내 시스템 사용 매뉴얼\n\n## 1. 계정 발급\n입사 후 IT팀에 계정 발급을 신청한다. 신청은 사내 포털의 IT 요청 메뉴에서 진행한다.\n\n## 2. VPN 접속\n사외에서 사내 시스템에 접속할 때는 VPN을 사용한다.\n\n## 3. 문서 저장 규칙\n업무 문서는 부서별 공유 드라이브에 저장한다.\n",
    "공용/규정/출장여비규정.md": "# 출장 여비 규정\n\n## 국내 출장\n국내 출장 시 일비 3만원, 숙박비 실비를 지급한다.\n\n## 해외 출장\n해외 출장 시 지역별 기준에 따라 차등 지급한다.\n",
    "임시/신규문서.md": "",
    "임시/템플릿.md": "# 제목\n\n작성자 :\n작성일 :\n부서 :\n\n내용을 입력하세요.\n",
}
DOCS_W1["임시/시스템매뉴얼_복사.md"] = DOCS_W1["공용/매뉴얼/시스템매뉴얼.md"]   # 같은 내용의 복사본

RULE_2026 = "# 임직원 복무 규정 (2026년판)\n\n## 제3조 (근무 시간)\n기본 근무 시간은 주 40시간으로 하며, 시업 시각은 오전 10시로 한다.\n\n## 제5조 (재택근무)\n임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다.\n\n## 제7조 (연차 휴가의 이월)\n당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 이월할 수 있다.\n"
SECURITY_0928 = DOCS_W1["공용/규정/보안지침.md"] + "\n## 3. 사내 AI 서비스\n사내에 설치된 AI 서비스에는 대외비를 뺀 업무 문서를 입력할 수 있다.\n"
TRIP_0928 = DOCS_W1["공용/규정/출장여비규정.md"].replace("일비 3만원", "일비 4만원")


def write_drive(files):
    for rel, text in files.items():
        p = DRIVE / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(text, encoding="utf-8")


write_drive(DOCS_W1)
print("9월 7일 목록", P.stage_catalog("2026-09-07"))

# 9월 14일 : 실행이 없었습니다. 목록도 남지 않았습니다

(DRIVE / "보관").mkdir(exist_ok=True)
(DRIVE / "공용/규정/복무규정_2024.md").rename(DRIVE / "보관/복무규정_2024.md")
write_drive({"공용/규정/복무규정_최종.md": RULE_2026})
print("9월 21일 목록", P.stage_catalog("2026-09-21"))

write_drive({"공용/규정/보안지침.md": SECURITY_0928, "공용/규정/출장여비규정.md": TRIP_0928})
print("9월 28일 목록", P.stage_catalog("2026-09-28"))

import json

import d68_pipeline as P
import d68_release as R

if not P.SNAPS.exists():
    print("지난 주들의 목록이 없습니다. 문서 생성 셀을 먼저 실행하세요")

QUESTIONS = [
    ("q01", "재택근무는 일주일에 몇 번까지 신청할 수 있습니까", "공용/규정/복무규정_최종.md"),
    ("q02", "재택근무 신청은 언제까지 누구의 승인을 받아야 합니까", "공용/규정/복무규정_최종.md"),
    ("q03", "출근 시각은 몇 시입니까", "공용/규정/복무규정_최종.md"),
    ("q04", "못 쓴 연차는 다음 해로 며칠까지 넘길 수 있습니까", "공용/규정/복무규정_최종.md"),
    ("q05", "고객 개인정보를 회사 밖으로 보내려면 누구 승인이 필요합니까", "공용/규정/보안지침.md"),
    ("q06", "사내에 설치된 AI 서비스에 업무 문서를 넣어도 됩니까", "공용/규정/보안지침.md"),
    ("q07", "보안 지침을 어기면 어떻게 됩니까", "공용/규정/보안지침.md"),
    ("q08", "회사 밖에서 사내 시스템에 접속하려면 무엇을 씁니까", "공용/매뉴얼/시스템매뉴얼.md"),
    ("q09", "입사하면 계정 발급은 어디서 신청합니까", "공용/매뉴얼/시스템매뉴얼.md"),
    ("q10", "업무 문서는 어디에 저장해야 합니까", "공용/매뉴얼/시스템매뉴얼.md"),
    ("q11", "국내 출장 일비는 얼마입니까", "공용/규정/출장여비규정.md"),
    ("q12", "해외 출장 여비는 어떻게 지급합니까", "공용/규정/출장여비규정.md"),
]
R.EVAL_SET.parent.mkdir(parents=True, exist_ok=True)
R.EVAL_SET.write_text(json.dumps([{"id": i, "question": q, "source": s} for i, q, s in QUESTIONS],
                                 ensure_ascii=False, indent=1), encoding="utf-8")

R.PROMPT.parent.mkdir(parents=True, exist_ok=True)
R.PROMPT.write_text("당신은 사내 규정 안내 도우미입니다. 주어진 규정 문서에 있는 내용으로만 답하고 "
                    "문서에 없으면 모른다고 답하십시오. 답변은 반드시 한국어로만 작성하세요.\n",
                    encoding="utf-8")

print(f"평가 질문 세트 {R.EVAL_SET.stem:<12} 해시 {P.fp(R.EVAL_SET.read_bytes())}")
print(f"프롬프트       {R.PROMPT.stem:<12} 해시 {P.fp(R.PROMPT.read_bytes())}")

# 화면 실습에서 재사용할 입력을 미리 고정합니다.
print("화면 실습 입력:", P.stage_catalog("2026-11-09"))

9월 7일 목록 /workspace/work_d68/snapshots/2026-09-07.json
9월 21일 목록 /workspace/work_d68/snapshots/2026-09-21.json
9월 28일 목록 /workspace/work_d68/snapshots/2026-09-28.json
평가 질문 세트 규정-질문-v2     해시 7001c8e4b717
프롬프트       규정-답변-v4     해시 2e6f9758297e
화면 실습 입력: /workspace/work_d68/snapshots/2026-11-09.json


In [23]:
import subprocess
from importlib.metadata import version

print("Airflow", version("apache-airflow"))
initialized = subprocess.run(
    [sys.executable, "-m", "airflow", "db", "migrate"],
    cwd=WORK, capture_output=True, text=True,
)
if initialized.returncode:
    print(initialized.stdout[-4000:], initialized.stderr[-4000:])
    initialized.check_returncode()
print("메타데이터 DB 준비 완료:", WORK / "airflow")

Airflow 3.3.2
메타데이터 DB 준비 완료: /workspace/work_d68/airflow


## 1. DAG 작성과 의존 관계 확인

### 1.1 실행 일정과 태스크 연결 작성

아래 셀을 실행하면 `dags/d68_weekly_refresh.py`가 저장됩니다. `@task`가 붙은 함수는 준비된 문서 처리 함수를 호출하되, 실행 순서와 실패 처리는 Airflow가 관리하도록 작성합니다.

`SCHEDULE`에는 한국 시각 매주 월요일 03:00에 해당하는 크론 표현식을, `RETRIES`에는 추가 재시도 횟수 2를 적으십시오. 이어서 세 개의 `None`을 태스크 호출로 바꾸어 `catalog → prepare → build_index → evaluate → gate → promote` 순서로 연결합니다. 파일을 저장한 뒤에는 다음 셀에서 실제 연결을 확인합니다.

태스크 호출의 반환값은 실행 결과 자체가 아니라 **XComArg**라는 참조입니다. 예를 들어 `prepare(snapshot)`은 `catalog`의 실행 결과를 기다린 뒤 그 경로를 입력으로 받도록 연결합니다. 이번 실습의 **XCom**에는 문서 전체 대신 파일 경로를 전달합니다.

같은 태스크 정의로 주간 일정 DAG와 화면 실습용 DAG를 만듭니다. 화면 실습용은 11월 9일 입력을 고정하고 운영 승격을 건너뛰며, UI에서 직접 Trigger합니다.

운영에 적용할 때 확인할 설정도 코드에 포함합니다. 한국 시간대와 논리적 날짜로 입력을 구분하고, 일시적인 연결 장애는 재시도하되 잘못된 설정과 품질 미달은 즉시 실패시킵니다. `max_active_runs=1`로 동시 실행을 제한하고 태스크마다 실행 시간 상한을 둡니다. 과거 입력이 없는 백필은 현재 문서로 대신 처리하지 않으며, 백필과 화면 연습은 운영 참조를 바꾸지 않습니다.

### 실무에 연결할 구현 위치

| 위치 | 예제에서 하는 일 | 실제 업무에 연결할 대상 |
|---|---|---|
| `P.stage_catalog` | 고정 시점의 문서 목록과 해시 보관 | 승인된 사내 문서 저장소의 수집 결과 |
| `P.prepare` | 정제·청크 생성과 입력 검증 | 기존 RAG 전처리 파이프라인 |
| `R.evaluate`·`gate` | 같은 질문으로 평가하고 품질 미달 차단 | 팀의 평가 세트와 배포 허용 기준 |
| `R.promote` | 검증한 번들로 운영 참조 변경 | 인덱스 별칭·버전 레지스트리의 전환 작업 |

문서 처리 구현이 바뀌어도 태스크의 입력·출력 경로와 실패 처리를 유지하면 같은 DAG 구조로 운영할 수 있습니다.

In [24]:
%%writefile dags/d68_weekly_refresh.py
from datetime import timedelta
import pendulum
from airflow.sdk import Param, dag, task
from airflow.sdk.exceptions import AirflowFailException, AirflowSkipException
import d68_pipeline as P
import d68_release as R

SCHEDULE = "0 3 * * 1"
RETRIES = 2
if SCHEDULE is None or RETRIES is None:
    raise ValueError("DAG 작성 셀의 SCHEDULE과 RETRIES를 완성한 뒤 다시 저장하십시오.")

def make_refresh(dag_id, schedule, rehearsal_day=None):
    parts_dir = P.WORK / "ops_rehearsal" if rehearsal_day else None
    @dag(
        dag_id=dag_id,
        schedule=schedule,
        is_paused_upon_creation=rehearsal_day is None,
        start_date=pendulum.datetime(2026, 9, 1, tz="Asia/Seoul"),
        catchup=False,
        max_active_runs=1,
        tags=["d68"],
        default_args={"retries": RETRIES, "retry_delay": timedelta(seconds=3),
                      "execution_timeout": timedelta(minutes=2)},
        params={"chunk_size": Param(400, type="integer"), "min_chars": Param(10, type="integer")},
    )
    def weekly_refresh():
        @task
        def catalog(**ctx):
            day = rehearsal_day or P.day_of(ctx["logical_date"])
            backfill = ctx["dag_run"].run_type == "backfill"
            try:
                return P.stage_catalog(day, allow_scan=not backfill and rehearsal_day is None)
            except P.MissingSnapshot as error:
                raise AirflowFailException(str(error)) from error
    
        @task
        def prepare(snapshot_path, **ctx):
            try:
                result = P.prepare(snapshot_path, ctx["params"], parts_dir=parts_dir)
            except P.BadConfig as error:
                raise AirflowFailException(str(error)) from error
            if not result["ok"]:
                raise AirflowFailException(result["message"])
            return result["chunks_path"]
    
        @task
        def build_index(chunks_path):
            return R.build_index(chunks_path)
    
        @task
        def evaluate(index_dir):
            return R.evaluate(index_dir, parts_dir=parts_dir)
    
        @task
        def gate(report_path, **ctx):
            if ctx["dag_run"].run_type == "backfill":
                raise AirflowSkipException("백필은 평가만 기록하고 운영 승격을 하지 않습니다.")
            ok, reason = R.decide_promotion(report_path)
            if not ok:
                raise AirflowFailException(f"승격 거절: {reason}. current는 유지합니다.")
            return report_path
    
        @task
        def promote(report_path, **ctx):
            if rehearsal_day:
                raise AirflowSkipException("화면 실습에서는 운영 번들을 유지합니다")
            release, changed = R.promote(report_path, airflow_run=ctx["run_id"])
            if not changed:
                raise AirflowSkipException(f"이미 같은 번들을 사용합니다: {release}")
            return release
    
        snapshot = catalog()
        chunks = prepare(snapshot)
        index_dir = build_index(chunks)
        report_path = evaluate(index_dir)
        checked_report = gate(report_path)
        if index_dir is None or report_path is None or checked_report is None:
            raise ValueError("DAG 작성 셀에서 세 태스크의 연결을 완성한 뒤 다시 저장하십시오.")
        promote(checked_report)

    return weekly_refresh()

dag_obj = make_refresh("d68_weekly_refresh", SCHEDULE)
ops_dag = make_refresh("d68_ops_rehearsal", None, rehearsal_day="2026-11-09")

Overwriting dags/d68_weekly_refresh.py


<details><summary>작성 예시 · 직접 연결한 뒤 확인하십시오</summary>

위 DAG 작성 셀에서 설정 두 줄과 연결 세 줄을 다음과 같이 수정합니다. 예시만 별도로 실행하지 말고 DAG 작성 셀을 수정한 뒤 다시 실행해 파일을 저장하십시오.

```python
SCHEDULE = "0 3 * * 1"
RETRIES = 2

index_dir = build_index(chunks)
report_path = evaluate(index_dir)
checked_report = gate(report_path)
```

</details>

### 1.2 저장한 DAG의 의존 관계 확인

아래 셀은 DAG 파일을 읽어 태스크별 선행·후속 작업과 재시도 설정을 출력합니다. 아직 문서 처리는 실행하지 않습니다. `gate`의 후속 작업이 `promote`인지 확인하고, 평가 전에 승격이 실행될 수 없는 이유를 연결 관계로 설명하십시오.

In [25]:
import importlib
import json

importlib.invalidate_caches()
if "d68_weekly_refresh" in sys.modules:
    del sys.modules["d68_weekly_refresh"]
try:
    dag_module = importlib.import_module("d68_weekly_refresh")
except Exception as error:
    raise RuntimeError(
        "DAG 파일을 읽지 못했습니다. 1.1절의 SCHEDULE·RETRIES와 세 태스크 연결을 "
        "모두 작성하고 DAG 작성 셀을 다시 실행하세요. 아래 원인 오류도 확인하세요."
    ) from error
dag_obj = dag_module.dag_obj
ORDER = ["catalog", "prepare", "build_index", "evaluate", "gate", "promote"]
GRAPH = []
for task_id in ORDER:
    item = dag_obj.get_task(task_id)
    row = {"task":task_id, "앞 작업":sorted(item.upstream_task_ids),
           "뒤 작업":sorted(item.downstream_task_ids), "retries":item.retries}
    GRAPH.append(row)
    print(row)
print("일정:", dag_module.SCHEDULE, "/ 시작일 시간대:", dag_obj.start_date.tzinfo)

print("DAG 파일:", DAGS / "d68_weekly_refresh.py")
print("이 파일의 DAG ID:", dag_module.dag_obj.dag_id, "/", dag_module.ops_dag.dag_id)

{'task': 'catalog', '앞 작업': [], '뒤 작업': ['prepare'], 'retries': 2}
{'task': 'prepare', '앞 작업': ['catalog'], '뒤 작업': ['build_index'], 'retries': 2}
{'task': 'build_index', '앞 작업': ['prepare'], '뒤 작업': ['evaluate'], 'retries': 2}
{'task': 'evaluate', '앞 작업': ['build_index'], '뒤 작업': ['gate'], 'retries': 2}
{'task': 'gate', '앞 작업': ['evaluate'], '뒤 작업': ['promote'], 'retries': 2}
{'task': 'promote', '앞 작업': ['gate'], '뒤 작업': [], 'retries': 2}
일정: 0 3 * * 1 / 시작일 시간대: UTC
DAG 파일: /workspace/work_d68/dags/d68_weekly_refresh.py
이 파일의 DAG ID: d68_weekly_refresh / d68_ops_rehearsal


## 2. 사전 점검: 정상 번들 한 번 실행

화면 실습에 앞서 `dag.test()`로 정상 번들을 한 번 실행해 설치·DAG 연결·평가 데이터가 준비됐는지 확인합니다. 다음 절에서는 `airflow standalone`을 기동하고 브라우저에서 직접 실행과 로그 확인, 태스크 초기화를 진행합니다.

`run_one.py`는 `dag_obj.test(...)`를 별도 프로세스에서 실행하는 점검 파일입니다. 아래 두 셀로 정상 번들을 만든 뒤 화면 실습으로 이동하십시오.

In [26]:
%%writefile run_one.py
import json
import sys
import pendulum
sys.path.insert(0, "dags")
from d68_weekly_refresh import dag_obj

logical_date = pendulum.parse(sys.argv[1], tz="Asia/Seoul").add(hours=3)
conf = json.loads(sys.argv[2])
run = dag_obj.test(logical_date=logical_date, run_conf=conf)
states = {}
values = {}
for item in run.get_task_instances():
    states[item.task_id] = {"state":str(item.state), "try_number":item.try_number}
    if str(item.state) == "success":
        values[item.task_id] = item.xcom_pull(task_ids=item.task_id, key="return_value")
print("RESULT " + json.dumps({"run_id":run.run_id, "state":str(run.state),
    "tasks":states, "xcom":values}, ensure_ascii=False))

Overwriting run_one.py


In [27]:
RUNS = []

def run_refresh(label, day, min_chars=10):
    result = subprocess.run(
        [sys.executable, "run_one.py", day, json.dumps({"min_chars":min_chars})],
        cwd=WORK, capture_output=True, text=True,
    )
    log = result.stdout + "\n" + result.stderr
    log_dir = WORK / "logs"
    log_dir.mkdir(exist_ok=True)
    log_path = log_dir / f"run_{len(RUNS)+1}.log"
    log_path.write_text(log, encoding="utf-8")
    rows = [line[7:] for line in log.splitlines() if line.startswith("RESULT ")]
    if not rows:
        print(log[-5000:])
        raise RuntimeError(f"Airflow 실행 결과를 읽지 못했습니다. 로그: {log_path}")
    record = json.loads(rows[-1])
    record.update(label=label, day=day, min_chars=min_chars, log=str(log_path))
    RUNS.append(record)
    print(label, "실행 상태:", record["state"])
    for task_id in ORDER:
        print(task_id, record["tasks"][task_id])
    print("원문 로그:", log_path)
    return record

BASELINE = run_refresh("정상 기준", "2026-09-28")
BASE_CURRENT = R.read_current()
print("운영 번들:", BASE_CURRENT["release"] if BASE_CURRENT else None)

정상 기준 실행 상태: success
catalog {'state': 'success', 'try_number': 1}
prepare {'state': 'success', 'try_number': 1}
build_index {'state': 'success', 'try_number': 1}
evaluate {'state': 'success', 'try_number': 1}
gate {'state': 'success', 'try_number': 1}
promote {'state': 'skipped', 'try_number': 1}
원문 로그: /workspace/work_d68/logs/run_1.log
운영 번들: r-20260928-fb4029


## 3. Airflow UI에서 실행·실패·재실행

### 3.1 Airflow 기동과 화면 연결

`airflow standalone`은 API 서버·스케줄러·DAG 프로세서 등 개인 실습에 필요한 구성 요소를 함께 띄웁니다. 아래 셀로 컨테이너 안에서 서버를 기동한 뒤, 로컬 PC의 브라우저에서 `http://localhost:8680/dags`를 엽니다. JupyterLab은 코드 작성에, 별도 Airflow 탭은 실행·로그·XCom 확인에 사용합니다.

컨테이너의 API 서버는 `0.0.0.0:8680`에서 연결을 받고, Docker는 로컬 PC의 `127.0.0.1:8680`에만 포트를 공개합니다. 따라서 브라우저에서는 `localhost` 주소를 사용하고, 태스크는 컨테이너 내부의 루프백 주소로 상태를 보고합니다. 서버가 기동된 뒤 두 DAG가 목록에 반영되기까지 잠시 걸릴 수 있습니다.

`d68_ops_rehearsal`은 「1.1」에서 작성한 `d68_weekly_refresh.py` 안에 함께 정의됩니다. 별도의 같은 이름 파일을 찾을 필요가 없습니다. 화면 실습용 DAG는 11월 9일 합성 입력을 고정하고 운영 승격을 건너뜁니다. 주간 일정용 `d68_weekly_refresh`는 일시 정지 상태로 둡니다.

In [28]:
import time
import signal
import socket
import urllib.request

AF_PORT = 8680  # Docker에서 연결한 포트와 같아야 합니다.
BASE = f"http://127.0.0.1:{AF_PORT}"
PUBLIC_UI = "http://localhost:8680/dags"
existing = globals().get("af_proc")
if existing is None or existing.poll() is not None:
    with socket.socket() as probe:
        try:
            probe.bind(("0.0.0.0", AF_PORT))
        except OSError as error:
            raise RuntimeError(
                "8680 포트가 이미 사용 중입니다. 노트북을 저장한 뒤 호스트 터미널의 "
                "d68_local 폴더에서 docker compose restart notebook을 실행하고 "
                "JupyterLab에 다시 접속하세요."
            ) from error

AF_ENV = dict(
    os.environ,
    AIRFLOW__API__PORT=str(AF_PORT),
    AIRFLOW__API__HOST="0.0.0.0",
    AIRFLOW__API__BASE_URL="http://localhost:8680",
    AIRFLOW__CORE__EXECUTION_API_SERVER_URL=f"{BASE}/execution/",
    AIRFLOW__CORE__SIMPLE_AUTH_MANAGER_ALL_ADMINS="True",
    AIRFLOW__CORE__DAG_RUN_CONF_OVERRIDES_PARAMS="True",
    AIRFLOW__LOGGING__LOGGING_LEVEL="INFO",
    AIRFLOW__DAG_PROCESSOR__REFRESH_INTERVAL="10",
    AIRFLOW__DAG_PROCESSOR__MIN_FILE_PROCESS_INTERVAL="10",
)
# 개인 실습의 UI이며, 호스트의 루프백 포트에만 연결합니다.
AF_ENV["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")

In [29]:
af_proc = globals().get("af_proc")


def start_airflow(timeout=240):
    """standalone 을 새 프로세스 그룹으로 기동하고 API 서버가 응답할 때까지 기다립니다"""
    global af_proc
    if af_proc is None or af_proc.poll() is not None:
        with (WORK / "standalone.log").open("w") as log:
            af_proc = subprocess.Popen([sys.executable, "-m", "airflow", "standalone"], env=AF_ENV, stdout=log,
                                       stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(f"{BASE}/api/v2/version", timeout=3) as r:
                if r.status == 200:
                    print(f"Airflow 준비 {time.time() - t0:.0f}초, 포트 {AF_PORT}")
                    return True
        except OSError:
            time.sleep(3)
    print("시간 안에 기동되지 않았습니다. standalone.log의 마지막 내용을 확인하세요.")
    print("\n".join((WORK / "standalone.log").read_text(errors="replace").splitlines()[-30:]))
    return False


def stop_airflow():
    """프로세스 그룹 전체를 종료합니다. 부모만 종료하면 스케줄러와 워커가 남습니다"""
    if af_proc is None or af_proc.poll() is not None:
        print("기동되어 있는 Airflow 가 없습니다")
        return
    os.killpg(af_proc.pid, signal.SIGTERM)
    try:
        af_proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        os.killpg(af_proc.pid, signal.SIGKILL)
    print("Airflow 를 종료했습니다")



AIRFLOW_UI_READY = start_airflow()

Airflow 준비 0초, 포트 8680


In [30]:
from IPython.display import HTML, display

if not AIRFLOW_UI_READY:
    raise RuntimeError("앞 셀의 Airflow 기동 오류를 해결한 뒤 다시 실행하세요.")
current = R.read_current()
if current is None:
    raise RuntimeError("2절의 정상 번들 사전 점검을 먼저 마치세요.")
OPS_CURRENT_BEFORE = current["release"]
print("로컬 PC의 브라우저에서 열 주소:", PUBLIC_UI)
display(HTML(
    f'<a href="{PUBLIC_UI}" target="_blank" rel="noopener">Airflow DAG 목록 열기</a>'
))
print("확인할 DAG: d68_ops_rehearsal / 주간 일정용 DAG는 일시 정지 상태로 유지")
print("DAG 정의 파일:", DAGS / "d68_weekly_refresh.py")

로컬 PC의 브라우저에서 열 주소: http://localhost:8680/dags


확인할 DAG: d68_ops_rehearsal / 주간 일정용 DAG는 일시 정지 상태로 유지
DAG 정의 파일: /workspace/work_d68/dags/d68_weekly_refresh.py


![로컬 Docker에서 실행한 Airflow DAG 목록입니다. 같은 파일에 정의한 d68_ops_rehearsal과 d68_weekly_refresh가 함께 표시됩니다.](lms-img:0)

### 3.2 Graph와 XCom으로 작성한 연결 확인

1. **Dags**에서 `d68_ops_rehearsal`을 엽니다. **Code**에서 저장한 DAG 파일을 확인하고 **Show Graph** 아이콘으로 여섯 태스크의 연결을 코드와 대조합니다. 소스를 수정했다면 저장 후 **Reparse Dag**로 다시 읽고 반영된 버전을 확인합니다. Graph는 연결을 확인하는 화면이며 정의의 편집은 파이썬 파일에서 합니다.
2. **Trigger**의 파라미터에서 `min_chars=150`을 입력해 실행합니다. 실행 ID를 기록한 뒤 **Grid** 또는 해당 실행의 **Graph**에서 `evaluate`·`gate`·`promote`의 상태를 읽습니다.
3. 성공한 `catalog` 또는 `prepare` 태스크를 열어 **Storage → XCom**에서 `return_value`의 파일 경로를 확인합니다. 화면 버전에 따라 **XCom** 탭으로 바로 표시될 수 있습니다. 파일 전체 대신 경로를 전달하는 이유를 적습니다.
4. 실패한 `gate`의 **Logs**를 열어 승격을 거절한 이유와 뒤집힌 질문 ID를 찾습니다. `promote`가 왜 실행되지 않았는지 의존 관계로 설명합니다.

목록이 비어 있으면 「1.2」에서 여섯 태스크와 두 DAG ID가 출력됐는지 확인합니다. DAG 작성 셀의 다섯 칸을 완성하지 않았으면 파일이 저장돼도 DAG를 등록할 수 없습니다. 오류를 해결해 파일을 다시 저장한 뒤 잠깐 기다리고 UI를 새로고침합니다. 접속이 안 되면 「3.1」의 기동 결과와 `work_d68/standalone.log`를 확인하십시오. 화면을 확인하지 못한 상태를 UI 실습 완료로 기록하지 않습니다.

![Trigger 창의 Run Parameters입니다. min_chars의 초기값은 10이며, 실패 실행에서는 150으로 수정합니다.](lms-img:0)

![UI에서 직접 실행한 Graph입니다. 앞의 네 작업은 성공했고 gate가 실패해 promote는 upstream_failed로 남았습니다.](lms-img:0)

![gate의 Logs에서 AirflowFailException과 뒤집힌 질문 q11·q12를 확인한 실제 화면입니다.](lms-img:0)

![성공한 태스크의 Storage 화면에서 XCom 반환 경로를 확인합니다. 경로가 가리키는 파일을 다음 태스크가 읽습니다.](lms-img:0)

### 3.3 실패한 gate만 Clear하고 원인 수정 후 새 실행

1. 실패 실행이 끝난 뒤 그 실행의 `gate`만 선택하고 **Clear**를 누릅니다. **Upstream**과 **Downstream**을 끄고 초기화 대상이 해당 실행의 `gate` 하나인지 확인한 뒤 실행합니다.
2. `catalog`부터 다시 실행됐는지, 같은 설정에서 `gate`가 다시 실패했는지 **Grid**와 **Logs**로 확인합니다. **Clear는 설정이나 데이터를 수정하지 않으므로** 품질 실패가 해결되지 않습니다.
3. **Trigger**로 새 실행을 만들고 `min_chars=10`으로 바꿉니다. 새 실행에서 `gate`가 통과하는지 확인합니다. 화면 실습용 `promote`는 운영 참조를 바꾸지 않도록 `skipped`가 됩니다.
4. 두 실행의 **Run Config**와 태스크 상태·로그를 비교해, 태스크 초기화와 원인 수정 후 새 실행의 차이를 적습니다.

`upstream_failed`는 앞 작업의 실패 때문에 실행되지 않은 상태이고, 여기의 `skipped`는 화면 실습에서 운영 변경을 하지 않도록 코드가 건너뛴 상태입니다.

![Downstream을 끈 뒤 Affected Tasks가 gate 하나인지 확인합니다. Confirm 전에 초기화 범위를 다시 읽습니다.](lms-img:0)

In [34]:
UI_RECORDS = [
    {
        "활동": "150자 새 실행",
        "run_id": "manual__2026-11-26T06:20:29.490432+00:00",
        "gate": "failed",
        "promote": "upstream_failed",
        "로그 근거": "gate 로그: AirflowFailException: 승격 거절: flip 2개 ['q11', 'q12']. current는 유지합니다.",
    },
    {
        "활동": "같은 실행의 gate만 Clear",
        "run_id": "manual__2026-11-26T06:20:29.490432+00:00",
        "gate": "failed",
        "promote": "upstream_failed",
        "로그 근거": "catalog~evaluate 재실행 없이 gate만 재시도(Try 2)됐으나, Run Config(min_chars=150)가 유지되어 동일하게 flip 2개 ['q11', 'q12']로 실패",
    },
    {
        "활동": "10자 새 실행",
        "run_id": "manual__2026-11-26T07:07:36.811066+00:00",
        "gate": "success",
        "promote": "skipped",
        "로그 근거": "gate 통과 후 promote 실행됨. promote 로그: AirflowSkipException: 화면 실습에서는 운영 번들을 유지합니다",
    },
]

AIRFLOW_NOTES = {
    "Graph에서 확인한 의존 관계": "catalog -> prepare -> build_index -> evaluate -> gate -> promote (직렬 파이프라인)",
    "XCom 경로와 전달 이유": "/workspace/work_d68/snapshots/2026-11-09.json — XCom은 메타데이터 DB에 저장되므로 대용량 파일 본문 대신 경로 문자열만 전달해 DB 부하와 직렬화·용량 제한 문제를 방지함",
    "Clear한 태스크 범위": "Upstream과 Downstream 옵션을 모두 끄고 해당 실행(manual__2026-11-26T06:20:29.490432+00:00)의 gate 태스크 1개만 Clear함",
    "Clear와 새 실행의 차이": "Clear는 기존 Run과 Run Config(min_chars=150)를 그대로 둔 채 태스크 상태만 초기화해 재실행하므로 설정/데이터 오류를 해결하지 못함. 반면 새 실행(Trigger)은 수정한 파라미터(min_chars=10)로 새로운 DAG Run을 처음부터 실행해 근본 원인을 해결함",
}

after_ui = R.read_current()["release"]
print("current:", OPS_CURRENT_BEFORE, "->", after_ui)
print("운영 번들 유지:", OPS_CURRENT_BEFORE == after_ui)

UI_COMPLETE = True  # Graph·Logs·Storage·Clear·새 실행을 직접 확인한 뒤 True로 바꿉니다.

current: r-20260928-fb4029 -> r-20260928-fb4029
운영 번들 유지: True


![같은 실행의 gate만 Clear한 뒤 다시 실패한 로그입니다. 태스크 초기화가 잘못된 정제 설정을 바꾸지는 않습니다.](lms-img:0)

![min_chars=10으로 만든 새 실행입니다. 평가 게이트는 통과하고 화면 연습의 promote는 의도적으로 skipped가 됩니다.](lms-img:0)

## 4. 화면에서 확인한 기록과 복구 범위 정리

UI에서 실행·로그 확인·Clear·새 실행을 모두 마친 뒤 `UI_RECORDS`와 `AIRFLOW_NOTES`를 채웁니다. 같은 실패 태스크만 다시 실행한 경우와 정제 기준을 고쳐 새 실행을 만든 경우를 구분하고, 수집 스냅샷의 재사용과 평가 재실행의 범위를 작성하십시오.

In [ ]:
for record in UI_RECORDS:
    print(record)
RECOVERY = {
    "재사용할 입력": "같은 입력 스냅샷(snapshots/YYYY-MM-DD.json, 실습 예: 2026-11-09.json)과 blobs/<sha256> 원문은 그대로 재사용함",
    "다시 계산할 결과": "정제 기준(min_chars 등)이 달라지면 청크(prepare)·인덱스(build_index)·평가(evaluate)를 새로 계산해 다시 확인함",
    "승격 조건": "gate에서 백필이 아니면서 평가 보고서가 정상이고, 첫 번들은 9/12 이상 적중 또는 기존 번들 대비 flip 0개여야 운영 승격 가능. 단, 화면 실습(d68_ops_rehearsal)에서는 promote를 건너뛰어(skipped) 실제 운영 반영은 하지 않음",
}

{'활동': '150자 새 실행', 'run_id': 'manual__2026-11-26T06:20:29.490432+00:00', 'gate': 'failed', 'promote': 'upstream_failed', '로그 근거': "gate 로그: AirflowFailException: 승격 거절: flip 2개 ['q11', 'q12']. current는 유지합니다."}
{'활동': '같은 실행의 gate만 Clear', 'run_id': 'manual__2026-11-26T06:20:29.490432+00:00', 'gate': 'failed', 'promote': 'upstream_failed', '로그 근거': "catalog~evaluate 재실행 없이 gate만 재시도(Try 2)됐으나, Run Config(min_chars=150)가 유지되어 동일하게 flip 2개 ['q11', 'q12']로 실패"}
{'활동': '10자 새 실행', 'run_id': 'manual__2026-11-26T07:07:36.811066+00:00', 'gate': 'success', 'promote': 'skipped', '로그 근거': 'gate 통과 후 promote 실행됨. promote 로그: AirflowSkipException: 화면 실습에서는 운영 번들을 유지합니다'}


<details><summary>기록 방향 · 실제 UI 결과를 먼저 작성하십시오</summary>

`AIRFLOW_NOTES`에는 직접 연 태스크의 XCom 경로와 로그, Clear 대상 범위를 적습니다. 실행 ID와 상태는 자신의 UI에서 확인한 값을 사용합니다.

`RECOVERY`는 같은 입력 스냅샷은 재사용하되 정제 기준이 달라지면 청크·인덱스·평가를 다시 확인한다는 원칙으로 작성합니다. 화면 실습에서는 승격 태스크를 건너뛰었으므로 실제 운영 반영을 완료했다고 적지 않습니다.

</details>

## 최종 판단과 제출

태스크 상태와 평가 결과, `current` 중 두 가지를 근거로 이번 후보를 운영에 반영할지 한 문장으로 작성하십시오. 짝이 같은 실행 기록을 읽고 판단을 재확인할 수 있도록 DAG 연결과 실행 인자도 함께 보관합니다. 실행하지 못한 항목은 미실행 또는 판정 보류로 적습니다.

In [38]:
DECISION = (
    "실행 기록 검증 결과, min_chars=10 기준(BASELINE 2026-09-28 실행 성공 및 "
    "d68_ops_rehearsal 10자 새 실행 gate=success, promote=skipped)에서 파이프라인과 품질 게이트가 정상 동작함을 확인함. "
    "반면 min_chars=150 오류 설정 시 출장여비규정 누락으로 flip 2개(['q11', 'q12'])가 발생해 gate가 AirFlowFailException으로 즉시 차단(promote=upstream_failed)했고, "
    "gate 단독 Clear(Try 2)는 기존 Run Config가 유지되어 동일하게 실패하므로 설정 오류 복구 시에는 동일 스냅샷을 재사용한 채 파라미터를 수정해 새 실행을 수행해야 함. "
    "따라서 현재 운영 번들(current.json)은 변경 없이 안전하게 유지하고, 운영 DAG(d68_weekly_refresh)의 기준 정제 설정은 min_chars=10으로 유지하여 운영함."
)

if UI_COMPLETE is not True or any(value is None for row in UI_RECORDS for value in row.values()):
    print("UI 활동과 실행 기록을 먼저 완성하십시오.")
elif DECISION is None:
    print("실행 기록을 근거로 최종 판단을 작성하십시오.")
else:
    submission = {
        "day": 68, "decision": DECISION, "graph": GRAPH,
        "airflow_notes": AIRFLOW_NOTES, "preflight_runs": RUNS, "ui_records": UI_RECORDS, "recovery": RECOVERY,
        "current": R.read_current(),
    }
    destination = WORK / "d68_submission.json"
    destination.write_text(json.dumps(submission, ensure_ascii=False, indent=2), encoding="utf-8")
    print("제출 파일:", destination)

제출 파일: /workspace/work_d68/d68_submission.json


## 종료와 참고

제출 셀과 아래의 `stop_airflow()`를 실행하고 노트북을 저장합니다. 로컬 실습 폴더에서 `work_d68/d68_submission.json`과 실행 로그를 확인한 뒤, 호스트 터미널의 `d68_local` 폴더에서 `docker compose down`을 실행합니다. 이 명령은 JupyterLab이 실행 중인 컨테이너도 종료하므로 노트북 셀에서 실행하지 않습니다. 로컬에 연결한 결과 파일은 남습니다.

이번에는 `dag.test()` 사전 점검과 UI의 수동 실행을 수행했습니다. 정기 스케줄 자동 실행·운영 백필·에셋 트리거는 별도 운영 검증이 필요한 범위입니다.

- [Airflow 3.3.2: Grid·Graph·Logs·XCom 화면](https://airflow.apache.org/docs/apache-airflow/3.3.2/ui.html)
- [Airflow 3.3.2 TaskFlow](https://airflow.apache.org/docs/apache-airflow/3.3.2/tutorial/taskflow.html)
- [Airflow 공식 Docker 이미지 확장](https://airflow.apache.org/docs/docker-stack/build.html)

In [ ]:
stop_airflow()

## 개념 그림

![Airflow 태스크의 의존 관계: 앞 태스크의 결과 경로를 다음 태스크에 전달합니다.](lms-img:0)

![평가 게이트 실패 시 승격 태스크는 실행되지 않고 current를 유지합니다.](lms-img:0)

![DAG의 작업 의존 관계와 입력·결과 파일의 식별](lms-img:0)

*DAG의 작업 의존 관계와 입력·결과 파일의 식별*